In [ ]:
# @title 1. Setup & Install
# !pip install git+https://github.com/geotab/mygeotab-python -q
# !pip install openpyxl pandas pytz -q

import time
import mygeotab
import getpass
import pandas as pd
import pytz
from datetime import datetime, time as dtime, timedelta
from bisect import bisect_left
from collections import defaultdict

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RESULT_LIMIT = 25000

def paginated_multi_call(api, requests, batch_size=100, delay_seconds=3):
    """Batch a large list of Get requests into groups of batch_size with a delay
    between each batch to stay under the 2000 calls/min API rate limit."""
    results = []
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i + batch_size]
        results.extend(api.multi_call(batch))
        if i + batch_size < len(requests):
            time.sleep(delay_seconds)
    return results

def fetch_all_by_date_chunks(api, type_name, search, chunk_days=3):
    """Fetch records by chunking the date range to bypass the 25k result limit.
    Deduplicates by event ID to handle records that appear at chunk boundaries.
    Prints a warning if any chunk hits the 25k limit (data may be truncated)."""
    start_dt = datetime.strptime(search['fromDate'], '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=pytz.UTC)
    end_dt   = datetime.strptime(search['toDate'],   '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=pytz.UTC)

    seen        = set()
    all_records = []
    chunk_start = start_dt

    while chunk_start < end_dt:
        chunk_end    = min(chunk_start + timedelta(days=chunk_days), end_dt)
        chunk_to_dt  = chunk_end - timedelta(seconds=1) if chunk_end < end_dt else chunk_end
        chunk_search = {**search,
            'fromDate': chunk_start.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'toDate':   chunk_to_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        }
        batch = api.call('Get', typeName=type_name, search=chunk_search) or []
        if len(batch) >= RESULT_LIMIT:
            print(f'WARNING: chunk {chunk_start.date()} → {chunk_to_dt.date()} returned {len(batch)} records '
                  f'(hit 25k limit — reduce chunk_days to avoid truncation)')
        for record in batch:
            rid = record.get('id')
            if rid not in seen:
                seen.add(rid)
                all_records.append(record)
        chunk_start = chunk_end

    return all_records

print('Setup complete.')

In [4]:
# @title 2. Authentication

USERNAME = 'farinnugraha'
DATABASE = 'smasmobility'

password = getpass.getpass(prompt='MyGeotab Password: ')
api = mygeotab.API(USERNAME, password, DATABASE)
api.authenticate()
print(f'Connected to: {DATABASE}')

Connected to: smasmobility


In [ ]:
# @title 3. Configuration

# Timezone
TZ = pytz.timezone('Asia/Jakarta')   # WIB (UTC+7)

# Date range (local time) -- change these
local_start = datetime(2026, 1, 1)
local_end   = datetime(2026, 1, 31)

# Convert to UTC ISO strings for the API
start_utc = TZ.localize(datetime.combine(local_start, dtime.min)).astimezone(pytz.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')
end_utc   = TZ.localize(datetime.combine(local_end,   dtime.max)).astimezone(pytz.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')

# Report settings
CO2_FACTOR_KG_PER_L  = 2.31   # kg CO2 per liter -- petrol default; change to 2.68 for diesel
BATTERY_LOW_THRESHOLD = 11.5  # volts -- cells below this are highlighted red

# Customer group filter — set to a Group ID to restrict the report to one customer.
# Leave empty ('') to include all customers.
# Run Cell 4 first to see a list of all available customer group IDs.
CUSTOMER_GROUP_ID = ''

print(f'Period : {local_start.date()} -> {local_end.date()}')
print(f'UTC    : {start_utc} -> {end_utc}')
print(f'CO2    : {CO2_FACTOR_KG_PER_L} kg/L | Battery threshold: {BATTERY_LOW_THRESHOLD} V')
print(f'Filter : {"All customers" if not CUSTOMER_GROUP_ID else f"Group {CUSTOMER_GROUP_ID}"}')

In [ ]:
# @title 4. Fetch Devices & Groups
import re

# Fetch all groups to build a name map
all_groups = api.call('Get', typeName='Group')
group_map  = {g['id']: g.get('name', '') for g in all_groups}
print(f'Found {len(all_groups)} groups.')

def extract_customer_names(group_ids):
    names = []
    for gid in group_ids:
        gname  = group_map.get(gid, '')
        matches = re.findall(r'\[([^\[\]]+)\]', gname)
        names.extend(matches)
    return ', '.join(names) if names else 'N/A'

# Print available customer groups so the user can pick a CUSTOMER_GROUP_ID in Cell 3.
customer_groups = {
    gid: re.findall(r'\[([^\[\]]+)\]', gname)[0]
    for gid, gname in group_map.items()
    if re.findall(r'\[([^\[\]]+)\]', gname)
}
print(f'\nAvailable customer groups ({len(customer_groups)}):')
for gid, cname in sorted(customer_groups.items(), key=lambda x: x[1]):
    marker = '  ← selected' if gid == CUSTOMER_GROUP_ID else ''
    print(f'  {gid:30s}  {cname}{marker}')

now_utc = datetime.now(pytz.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')

# All devices active during the report period (includes since-archived ones)
devices_raw    = api.call('Get', typeName='Device', search={'fromDate': start_utc})
# Currently active devices — used to identify which were archived after the period
devices_active = api.call('Get', typeName='Device', search={'fromDate': now_utc})
active_ids     = {d['id'] for d in devices_active}

# Apply customer group filter if specified in Cell 3.
if CUSTOMER_GROUP_ID:
    before = len(devices_raw)
    devices_raw = [d for d in devices_raw
                   if any(g['id'] == CUSTOMER_GROUP_ID for g in (d.get('groups') or []))]
    print(f'\nCustomer filter applied: {before} → {len(devices_raw)} devices '
          f'(group: {customer_groups.get(CUSTOMER_GROUP_ID, CUSTOMER_GROUP_ID)})')

device_map = {}
for d in devices_raw:
    group_ids = [g['id'] for g in (d.get('groups') or [])]
    device_map[d['id']] = {
        'name':     d.get('name', ''),
        'vin':      d.get('vehicleIdentificationNumber', ''),
        'serial':   d.get('serialNumber', ''),
        'customer': extract_customer_names(group_ids),
    }

device_ids   = list(device_map.keys())
archived_ids = [did for did in device_ids if did not in active_ids]

# Build name → [device_ids] map; active device listed before archived (handles device swaps)
name_to_dids = defaultdict(list)
for did in ([d for d in device_ids if d in active_ids] + archived_ids):
    name_to_dids[device_map[did]['name']].append(did)

# Secondary merge-by-VIN: if two name groups share the same non-empty VIN, merge the
# later group into the first (canonical) one. Handles device swaps where the two entries
# have slightly different names in MYG (e.g. "Innova Zenix" vs "Toyota Innova Zenix").
vin_to_canonical = {}
to_merge = {}
for name, dids in name_to_dids.items():
    for did in dids:
        vin = (device_map[did].get('vin') or '').strip()
        if not vin:
            continue
        if vin in vin_to_canonical:
            canonical = vin_to_canonical[vin]
            if canonical != name and name not in to_merge:
                to_merge[name] = canonical
        else:
            vin_to_canonical[vin] = name

for name, canonical in to_merge.items():
    name_to_dids[canonical].extend(name_to_dids.pop(name))

if to_merge:
    print(f'VIN merge: collapsed {len(to_merge)} duplicate group(s): {list(to_merge.keys())}')

print(f'Found {len(device_ids)} devices ({len(archived_ids)} archived). Vehicle rows: {len(name_to_dids)}.')

In [ ]:
# @title 5. Fetch Trips & Aggregate (Speed)

def parse_duration(d):
    """Parse TimeSpan strings (HH:MM:SS or D.HH:MM:SS), timedelta, or None to total seconds."""
    if d is None:
        return 0.0
    if isinstance(d, timedelta):
        return d.total_seconds()
    if isinstance(d, str) and d:
        try:
            days, time_str = (d.split('.', 1) if '.' in d.split(':')[0] else (0, d))
            parts = time_str.split(':')
            if len(parts) == 3:
                return int(days) * 86400 + int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        except (ValueError, IndexError):
            pass
    return 0.0

# Fetch trips per device to bypass the 25k result limit
trip_requests = [['Get', {'typeName': 'Trip', 'search': {
    'deviceSearch': {'id': did},
    'fromDate': start_utc,
    'toDate':   end_utc,
}}] for did in device_ids]

trip_results = paginated_multi_call(api, trip_requests, batch_size=100)
trips_raw = [t for sublist in trip_results for t in (sublist or [])]
print(f'Fetched {len(trips_raw)} trips.')

trip_agg = defaultdict(lambda: {
    'max_speed':            0.0,
    'total_weighted_speed': 0.0,
    'total_drive_sec':      0.0,
})

raw_trips_for_sheet2 = []

for t in trips_raw:
    did     = t['device']['id']
    agg     = trip_agg[did]
    max_s   = float(t.get('maximumSpeed') or 0)
    avg_s   = float(t.get('averageSpeed') or 0)
    stop_dt = t.get('stop') or t.get('nextTripStart')
    drive_s = parse_duration(t.get('drivingDuration'))
    idle_s  = parse_duration(t.get('idlingDuration'))

    agg['max_speed']             = max(agg['max_speed'], max_s)
    agg['total_weighted_speed'] += avg_s * drive_s
    agg['total_drive_sec']      += drive_s

    raw_trips_for_sheet2.append({
        'did':       did,
        'start':     t.get('start'),
        'stop':      stop_dt,
        'drive_sec': drive_s,
        'idle_sec':  idle_s,
        'distance':  float(t.get('distance') or 0),
        'max_speed': max_s,
        'avg_speed': avg_s,
        'odo_km':    float(t.get('odometer') or 0) / 1000,
    })

print('Trip aggregation complete.')

In [ ]:
# @title 6. Fetch Odometer (StatusData — DiagnosticOdometerAdjustmentId)

odo_requests = [['Get', {'typeName': 'StatusData', 'search': {
    'deviceSearch':     {'id': did},
    'diagnosticSearch': {'id': 'DiagnosticOdometerAdjustmentId'},
    'fromDate': end_utc,
    'toDate':   end_utc,
}}] for did in device_ids]

odo_results = paginated_multi_call(api, odo_requests, batch_size=100)

odometer_km = {}
for did, records in zip(device_ids, odo_results):
    if records:
        latest = max(records, key=lambda r: r['dateTime'])
        odometer_km[did] = round(float(latest.get('data') or 0) / 1000, 2)
    else:
        odometer_km[did] = None

present = sum(1 for v in odometer_km.values() if v is not None)
print(f'Odometer data fetched for {present} of {len(device_ids)} devices.')

In [ ]:
# @title 7. Fetch Fill-ups & Fuel Consumed

fillup_count  = defaultdict(int)
fillup_volume = defaultdict(float)
try:
    fillup_requests = [['Get', {'typeName': 'FillUp', 'search': {
        'deviceSearch': {'id': did},
        'fromDate': start_utc,
        'toDate':   end_utc,
    }}] for did in device_ids]

    fillup_results = paginated_multi_call(api, fillup_requests, batch_size=100)
    fillup_raw = [f for sublist in fillup_results for f in (sublist or [])]

    for f in fillup_raw:
        did     = (f.get('device') or {}).get('id')
        vol     = float(f.get('volume') or 0)
        derived = float(f.get('derivedVolume') or 0)
        # Mirror the exact precedence used by MYG's FillUpRecord.Volume property:
        #   WITH fuel card transaction → prefer volume (card-reported), fall back to derivedVolume
        #   WITHOUT fuel card transaction → prefer derivedVolume (OBD-estimated), fall back to volume
        has_transaction = 'FuelTransaction' in str(f.get('confidence') or '')
        if has_transaction:
            effective_vol = vol if vol > 0 else derived
        else:
            effective_vol = derived if derived > 0 else vol
        if did:
            fillup_count[did]  += 1
            fillup_volume[did] += effective_vol
    print(f'FillUp records: {len(fillup_raw)}')
except Exception as e:
    print(f'FillUp not available: {e}')

# Engine-reported fuel consumption — active devices only, matching MYG's fuel report scope.
# Archived devices and old device IDs from swapped vehicles are excluded; they appear as N/A.
fuel_consumed = defaultdict(float)
try:
    fuel_requests = [['Get', {'typeName': 'FuelUsed', 'search': {
        'deviceSearch': {'id': did},
        'fromDate': start_utc,
        'toDate':   end_utc,
    }}] for did in device_ids if did in active_ids]

    fuel_results  = paginated_multi_call(api, fuel_requests, batch_size=100)
    fuel_used_raw = [f for sublist in fuel_results for f in (sublist or [])]

    for f in fuel_used_raw:
        did = (f.get('device') or {}).get('id')
        if did:
            fuel_consumed[did] = round(fuel_consumed[did] + float(f.get('totalFuelUsed') or 0), 4)
    print(f'FuelUsed records: {len(fuel_used_raw)}')
except Exception as e:
    print(f'FuelUsed not available: {e}')

In [ ]:
# @title 8. Fetch Exception Events (Known Rule IDs)

def event_duration_sec(e):
    d = parse_duration(e.get('duration'))
    if d > 0:
        return d
    af = e.get('activeFrom')
    at = e.get('activeTo')
    if af and at:
        try:
            return max(0.0, (pd.to_datetime(at) - pd.to_datetime(af)).total_seconds())
        except Exception:
            pass
    return 0.0

# chunk_days=3 keeps each chunk well under 25k even at 2000 devices
idle_evts  = fetch_all_by_date_chunks(api, 'ExceptionEvent', {
    'ruleSearch': {'id': 'RuleIdlingId'},
    'fromDate': start_utc, 'toDate': end_utc,
}, chunk_days=3)

minor_evts = fetch_all_by_date_chunks(api, 'ExceptionEvent', {
    'ruleSearch': {'id': 'RuleEnhancedMinorCollisionId'},
    'fromDate': start_utc, 'toDate': end_utc,
}, chunk_days=3)

major_evts = fetch_all_by_date_chunks(api, 'ExceptionEvent', {
    'ruleSearch': {'id': 'RuleEnhancedMajorCollisionId'},
    'fromDate': start_utc, 'toDate': end_utc,
}, chunk_days=3)

speed_evts = fetch_all_by_date_chunks(api, 'ExceptionEvent', {
    'ruleSearch': {'id': 'RulePostedSpeedingId'},
    'fromDate': start_utc, 'toDate': end_utc,
}, chunk_days=3)

# Supplemental per-device queries for archived device IDs.
# The global ruleSearch query silently excludes events from archived devices;
# querying each archived device explicitly bypasses that filter.
if archived_ids:
    def supplemental_fetch(rule_id):
        reqs = [['Get', {'typeName': 'ExceptionEvent', 'search': {
            'deviceSearch': {'id': did},
            'ruleSearch':   {'id': rule_id},
            'fromDate': start_utc,
            'toDate':   end_utc,
        }}] for did in archived_ids]
        return [e for batch in paginated_multi_call(api, reqs, batch_size=100) for e in (batch or [])]

    def merge_into(base, extra):
        seen  = {e.get('id') for e in base}
        added = 0
        for e in extra:
            if e.get('id') not in seen:
                seen.add(e.get('id'))
                base.append(e)
                added += 1
        return added

    ia = merge_into(idle_evts,  supplemental_fetch('RuleIdlingId'))
    ma = merge_into(minor_evts, supplemental_fetch('RuleEnhancedMinorCollisionId'))
    ja = merge_into(major_evts, supplemental_fetch('RuleEnhancedMajorCollisionId'))
    sa = merge_into(speed_evts, supplemental_fetch('RulePostedSpeedingId'))
    print(f'Archived supplemental: +{ia} idling, +{ma} minor, +{ja} major, +{sa} speeding')

# Aggregate
idling_counts   = defaultdict(int)
idling_duration = defaultdict(float)
for e in idle_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        idling_counts[did]   += 1
        idling_duration[did] += event_duration_sec(e)

minor_counts = defaultdict(int)
major_counts = defaultdict(int)
for e in minor_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        minor_counts[did] += 1
for e in major_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        major_counts[did] += 1

speeding_counts = defaultdict(int)
for e in speed_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        speeding_counts[did] += 1

print(f'Idling events    : {len(idle_evts)}')
print(f'Minor collisions : {len(minor_evts)}')
print(f'Major collisions : {len(major_evts)}')
print(f'Speeding events  : {len(speed_evts)}')


In [ ]:
# @title 9. Fetch Battery Voltage (DiagnosticGoDeviceVoltageId)

# Only query active devices — archived devices have no current voltage; querying them
# with fromDate=now_utc returns their last-ever reading, which is stale and misleading.
active_device_ids = [did for did in device_ids if did in active_ids]

batt_requests = [['Get', {'typeName': 'StatusData', 'search': {
    'deviceSearch':     {'id': did},
    'diagnosticSearch': {'id': 'DiagnosticGoDeviceVoltageId'},
    'fromDate': now_utc,
    'toDate':   now_utc,
}}] for did in active_device_ids]

batt_results = paginated_multi_call(api, batt_requests, batch_size=100)

battery_voltage = {}
for did, records in zip(active_device_ids, batt_results):
    if records:
        latest = max(records, key=lambda r: r['dateTime'])
        battery_voltage[did] = round(float(latest.get('data') or 0), 2)
    else:
        battery_voltage[did] = None

low_count = sum(1 for v in battery_voltage.values() if v is not None and v < BATTERY_LOW_THRESHOLD)
present   = sum(1 for v in battery_voltage.values() if v is not None)
print(f'Battery voltage fetched for {present} active devices. Below {BATTERY_LOW_THRESHOLD} V: {low_count}')
print(f'({len(archived_ids)} archived devices set to N/A)')

In [ ]:
# @title 10. Build DataFrames & Preview

def fmt_duration(total_seconds):
    s = int(total_seconds or 0)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}'

def fmt_dt(dt):
    if not dt:
        return ''
    return pd.to_datetime(dt).tz_convert(TZ).strftime('%Y-%m-%d %H:%M:%S')

# -- Sheet 1: Vehicle Summary --
# Aggregate across all device IDs per vehicle name to handle device swaps and archived devices.
summary_rows = []
for name, dids in name_to_dids.items():
    info    = device_map[dids[0]]
    max_spd = max((trip_agg.get(d, {}).get('max_speed', 0) for d in dids), default=0)

    odo_vals = [odometer_km[d] for d in dids if odometer_km.get(d) is not None]
    odo      = max(odo_vals) if odo_vals else None

    fuel_L  = round(sum(fuel_consumed.get(d, 0) for d in dids), 2)
    co2_kg  = round(fuel_L * CO2_FACTOR_KG_PER_L, 2) if fuel_L > 0 else 'N/A'

    f_count = sum(fillup_count.get(d, 0)  for d in dids)
    f_vol   = sum(fillup_volume.get(d, 0) for d in dids)
    # Show N/A when fill-ups were detected but the API has no volume data (no fuel card).
    # This distinguishes "volume unknown" from "zero volume dispensed".
    f_vol_display = 'N/A' if (f_count > 0 and f_vol < 0.001) else int(round(f_vol))

    minor   = sum(minor_counts.get(d, 0)    for d in dids)
    major   = sum(major_counts.get(d, 0)    for d in dids)
    idle_c  = sum(idling_counts.get(d, 0)   for d in dids)
    idle_d  = sum(idling_duration.get(d, 0) for d in dids)
    spd_c   = sum(speeding_counts.get(d, 0) for d in dids)

    batt_v  = next((battery_voltage[d] for d in dids if battery_voltage.get(d) is not None), None)

    summary_rows.append({
        'Vehicle Name':        info['name'],
        'Customer':            info['customer'],
        'VIN':                 info['vin'] or 'N/A',
        'Serial No.':          info['serial'] or 'N/A',
        'Odometer (km)':       int(round(odo)) if odo is not None else 'N/A',
        'Fill-up Count':       f_count,
        'Fill-up Volume (L)':  f_vol_display,
        'Fuel Consumed (L)':   fuel_L if fuel_L > 0 else 'N/A',
        'Max Speed (km/h)':    round(max_spd, 1),
        'CO2 Emission (kg)':   co2_kg,
        'Minor Collisions':    minor,
        'Major Collisions':    major,
        'Idle Event Count':    idle_c,
        'Total Idling':        fmt_duration(idle_d),
        'Speeding Events':     spd_c,
        'Battery Voltage (V)': round(batt_v, 2) if batt_v is not None else 'N/A',
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.head(10))

# -- Sheet 2: Trip Detail --
trip_rows = []
for t in raw_trips_for_sheet2:
    info = device_map.get(t['did'], {})
    trip_rows.append({
        'Vehicle Name':      info.get('name', ''),
        'Customer':          info.get('customer', ''),
        'VIN':               info.get('vin', '') or 'N/A',
        'Trip Start':        fmt_dt(t['start']),
        'Trip End':          fmt_dt(t['stop']),
        'Duration':          fmt_duration(t['drive_sec']),
        'Distance (km)':     round(t['distance'], 2),
        'Avg Speed (km/h)':  round(t['avg_speed'], 1),
        'Max Speed (km/h)':  round(t['max_speed'], 1),
        'Idling Duration':   fmt_duration(t['idle_sec']),
        'Odometer End (km)': int(round(t['odo_km'])),
    })

trips_df = pd.DataFrame(trip_rows)
display(trips_df.head(10))
print(f'Summary: {len(summary_df)} vehicles | Trip detail: {len(trips_df)} trips')

In [ ]:
# @title 11. Generate Styled Excel Report

# Style constants (navy theme matching Geotab branding)
HEADER_BG  = '1F3864'
HEADER_FG  = 'FFFFFF'
ALT_ROW_BG = 'EBF3FB'
BAD_BG     = 'FCE4D6'   # red tint for battery low warning
BAD_FG     = 'C00000'   # dark red
GREY_FG    = '808080'   # grey for N/A cells
BLACK_FG   = '000000'

def _font(bold=False, color=BLACK_FG, size=10):
    return Font(name='Calibri', bold=bold, color=color, size=size)

def _fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def _border():
    s = Side(style='thin', color='BFBFBF')
    return Border(left=s, right=s, top=s, bottom=s)

def _align(h='left'):
    return Alignment(horizontal=h, vertical='center')

def write_header(ws, headers):
    for col, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.font      = _font(bold=True, color=HEADER_FG)
        c.fill      = _fill(HEADER_BG)
        c.alignment = _align('center')
        c.border    = _border()
    ws.row_dimensions[1].height = 22

def auto_widths(ws, headers, min_w=10, max_w=35):
    for col, h in enumerate(headers, 1):
        ltr = get_column_letter(col)
        content_w = max(
            (len(str(ws.cell(r, col).value or '')) for r in range(1, ws.max_row + 1)),
            default=len(h)
        )
        ws.column_dimensions[ltr].width = min(max(content_w + 3, min_w), max_w)

wb  = openpyxl.Workbook()
ws1 = wb.active
ws1.title = 'Vehicle Summary'
ws2 = wb.create_sheet('Trip Detail')

# -- Sheet 1: Vehicle Summary --
s1_cols = [c for c in summary_df.columns if c != '_did']
write_header(ws1, s1_cols)
ws1.freeze_panes = 'A2'

for row_i, (_, row) in enumerate(summary_df.iterrows(), start=2):
    bg = ALT_ROW_BG if row_i % 2 == 0 else 'FFFFFF'
    for col_i, col_name in enumerate(s1_cols, 1):
        val = row[col_name]
        c   = ws1.cell(row=row_i, column=col_i, value=val)
        c.border    = _border()
        c.alignment = _align()
        if col_name == 'Battery Voltage (V)':
            try:
                if float(val) < BATTERY_LOW_THRESHOLD:
                    c.fill = _fill(BAD_BG)
                    c.font = _font(bold=True, color=BAD_FG)
                else:
                    c.fill = _fill(bg)
                    c.font = _font()
            except (TypeError, ValueError):
                c.fill = _fill(bg)
                c.font = _font(color=GREY_FG)
        else:
            c.fill = _fill(bg)
            c.font = _font()

auto_widths(ws1, s1_cols)

# -- Sheet 2: Trip Detail --
s2_cols = list(trips_df.columns)
write_header(ws2, s2_cols)
ws2.freeze_panes = 'A2'

for row_i, (_, row) in enumerate(trips_df.iterrows(), start=2):
    bg = ALT_ROW_BG if row_i % 2 == 0 else 'FFFFFF'
    for col_i, col_name in enumerate(s2_cols, 1):
        c = ws2.cell(row=row_i, column=col_i, value=row[col_name])
        c.fill      = _fill(bg)
        c.font      = _font()
        c.border    = _border()
        c.alignment = _align()

auto_widths(ws2, s2_cols)

# Save
file_start = local_start.strftime('%Y-%m-%d')
file_end   = local_end.strftime('%Y-%m-%d')
filename   = f'SMAS_Mobility_Fleet_Report_{DATABASE}_{file_start}_to_{file_end}.xlsx'
wb.save(filename)

print(f'Report saved: {filename}')
print(f'  Sheet 1 - Vehicle Summary : {len(summary_df)} rows')
print(f'  Sheet 2 - Trip Detail     : {len(trips_df)} rows')

try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print(f'(Running locally -- file saved as {filename})')